# Credit Risk Score Pipeline — v2

**Goal:** Identify the best two-model combination for credit risk scoring by:
1. Simulating a synthetic portfolio (~25% charge-off rate)
2. Benchmarking 5 candidate models individually (Lorenz curves + Gini)
3. Selecting the best model pair via **JMI** (total combined information) with **CMI** shown as a diagnostic reference — both computed via 5-fold CV
4. Building a 5×5 score grid and assigning risk tiers using the 1.5× CO rate rule

See `README.md` for a summary and `METHODOLOGY.md` for detailed design rationale.

## 0 · Imports & Configuration

In [ ]:
import numpy as np
import pandas as pd
from itertools import combinations
from sklearn.model_selection import KFold
from sklearn.metrics import roc_auc_score
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import matplotlib.ticker as mticker
import warnings
warnings.filterwarnings("ignore")

%matplotlib inline
plt.rcParams.update({"figure.dpi": 130, "font.size": 9})
np.random.seed(42)

# ── Global parameters ─────────────────────────────────────────────────────────
N       = 50_000   # accounts to simulate
N_BINS  = 5        # quantile bins per model for the score grid and JMI discretisation
N_FOLDS = 5        # CV folds for JMI estimation
MULT    = 1.5      # minimum tier-over-tier CO rate multiplier
N_TIERS = 5        # ← set to 5, 6, or 7 (names and colours are pre-defined for all three)

TARGET = "charge_off"   # ← customise: column name of the binary (0/1) outcome variable

# ── Score direction ────────────────────────────────────────────────────────────
# +1 = higher score → safer   (bureau scores: FICO, VantageScore, internal scorecards)
# -1 = higher score → riskier (probability-of-default models, raw PD outputs in [0, 1])
# All scores are normalised to "higher = safer" before analysis (Section 1).
SCORE_DIRECTION = {
    "Model_A": +1,   # simulated bureau-style score — change to -1 for PD models
    "Model_B": +1,
    "Model_C": +1,
    "Model_D": +1,
    "Model_E": +1,
}

MODELS = list(SCORE_DIRECTION.keys())

# ── Risk tier names & colours (supports 5, 6, or 7 tiers) ────────────────────
# Full list of 7 names ordered safest → riskiest.
# N_TIERS controls how many are used; the first N_TIERS entries are active.
_ALL_TIER_NAMES = [
    "Very Low Risk",   # Tier 1 — safest
    "Low Risk",        # Tier 2
    "Moderate",        # Tier 3
    "Medium-High",     # Tier 4
    "High Risk",       # Tier 5
    "Very High Risk",  # Tier 6
    "Severe Risk",     # Tier 7 — riskiest
]

_ALL_TIER_COLORS = [
    "#2ecc71",  # bright green    — Very Low Risk
    "#a8e063",  # light green     — Low Risk
    "#d4e157",  # yellow-green    — Moderate
    "#f9c74f",  # amber           — Medium-High
    "#fd8c04",  # orange          — High Risk
    "#f76c1b",  # dark orange     — Very High Risk
    "#e03131",  # red             — Severe Risk
]

# Active slices (driven by N_TIERS — change only N_TIERS above)
TIER_ORDER   = _ALL_TIER_NAMES[:N_TIERS]
TIER_PALETTE = dict(zip(_ALL_TIER_NAMES, _ALL_TIER_COLORS))   # all 7 always registered

print(f"Configuration loaded — {N_TIERS} risk tiers active: {TIER_ORDER}")

## 1 · Simulate Dataset

A single **latent risk factor** `Z ~ N(0,1)` drives both the charge-off outcome and all five model scores.

- **Charge-off probability:** `p = sigmoid(1.5·Z − 1.5)` → overall CO rate ≈ 25%
- **Model score:** `S = −signal·Z + noise`, scaled to [300, 850]
- **Missingness:** 5% random per model score

| Model | Signal | Noise | Character |
|---|---|---|---|
| Model_A | 2.5 | 0.6 | Strong |
| Model_B | 2.2 | 0.8 | Strong (correlated with A) |
| Model_C | 1.6 | 1.2 | Moderate |
| Model_D | 1.1 | 1.6 | Weak |
| Model_E | 0.4 | 2.2 | Near-noise (complementary to A) |

**Score direction normalisation:** after simulation (or data load), any model with `SCORE_DIRECTION = -1` — i.e. a **probability-of-default** score where higher = riskier — is **negated** so the whole pipeline can uniformly assume *higher score = safer*. Bureau scores (FICO, VantageScore) and internal scorecards with `SCORE_DIRECTION = +1` are left unchanged. Set directions in the `SCORE_DIRECTION` dict in Section 0.

In [ ]:
def simulate_dataset(n: int) -> pd.DataFrame:
    latent = np.random.normal(0, 1, n)
    prob   = 1 / (1 + np.exp(-(1.5 * latent - 1.5)))
    co     = np.random.binomial(1, prob, n)

    def score(signal, noise):
        raw  = -signal * latent + np.random.normal(0, noise, n)
        norm = (raw - raw.min()) / (raw.max() - raw.min())
        return np.round(300 + norm * 550).astype(int)

    df = pd.DataFrame({
        "account_id": np.arange(1, n + 1),
        TARGET: co,
        "Model_A": score(2.5, 0.6),
        "Model_B": score(2.2, 0.8),
        "Model_C": score(1.6, 1.2),
        "Model_D": score(1.1, 1.6),
        "Model_E": score(0.4, 2.2),
    })
    for col in MODELS:
        df.loc[np.random.rand(n) < 0.05, col] = np.nan
    return df


df = simulate_dataset(N)
print(f"Accounts simulated  : {len(df):,}")
print(f"Target column       : '{TARGET}'")
print(f"Target event rate   : {df[TARGET].mean():.2%}")
print(f"Missing values per model:")
print(df[MODELS].isna().sum().to_string())
df.head()

In [ ]:
# ── Normalise score direction ──────────────────────────────────────────────────
# PD models (SCORE_DIRECTION = -1) have their score column negated so that the
# entire pipeline can uniformly assume "higher score = safer account".
# Bureau scores (FICO, VantageScore) and scorecard models are left unchanged.
for col, direction in SCORE_DIRECTION.items():
    if col in df.columns and direction == -1:
        df[col] = -df[col]

inverted = [c for c, d in SCORE_DIRECTION.items() if d == -1 and c in df.columns]
if inverted:
    print(f"Direction-normalised (negated) : {inverted}")
    print("These columns now show negative values; higher (less-negative) = safer.")
else:
    print("Direction normalisation        : none — all scores are already higher = safer.")

## 2 · Lorenz Curves & Gini Coefficients

The Lorenz curve (Cumulative Accuracy Profile) plots:
- **X:** cumulative % of accounts sorted riskiest → safest (score ascending)
- **Y:** cumulative % of charge-offs captured

**Gini = 2 × ROC_AUC − 1** (standard credit risk metric; computed via `roc_auc_score`, *not* from the CAP area which gives incorrect results at non-50% bad rates).

In [ ]:
def plot_lorenz_curves(df: pd.DataFrame) -> dict:
    """Plot Lorenz curves for all models and return {model: gini} dict."""
    fig, ax = plt.subplots(figsize=(8, 6))

    line_styles = ["-", "--", "-.", ":", (0, (3, 1, 1, 1))]
    colors      = ["#2c7bb6", "#d7191c", "#1a9641", "#fdae61", "#9e4fb5"]
    gini_scores = {}

    for i, model in enumerate(MODELS):
        sub = df[[TARGET, model]].dropna()
        sub = sub.sort_values(model, ascending=True).reset_index(drop=True)

        cum_accts = np.arange(1, len(sub) + 1) / len(sub)
        cum_co    = sub[TARGET].cumsum() / sub[TARGET].sum()

        # All scores are direction-normalised: higher = safer → negate for roc_auc_score
        gini = round(2 * roc_auc_score(sub[TARGET], -sub[model]) - 1, 3)
        gini_scores[model] = gini
        ax.plot(cum_accts, cum_co,
                linestyle=line_styles[i], color=colors[i], linewidth=1.8,
                label=f"{model}  (Gini = {gini:.3f})")

    ax.plot([0, 1], [0, 1], "k--", linewidth=1, label="Random model")
    ax.set_xlabel("Cumulative % of Accounts  (sorted riskiest → safest)")
    ax.set_ylabel(f"Cumulative % of '{TARGET}' Events Captured")
    ax.set_title("Lorenz Curves — Model Comparison", fontsize=12, fontweight="bold")
    ax.legend(loc="lower right", fontsize=9)
    ax.xaxis.set_major_formatter(mticker.PercentFormatter(xmax=1))
    ax.yaxis.set_major_formatter(mticker.PercentFormatter(xmax=1))
    ax.grid(True, alpha=0.3, linestyle="--")
    ax.set_xlim(0, 1); ax.set_ylim(0, 1)
    plt.tight_layout()
    plt.savefig("lorenz_curves.png", bbox_inches="tight")
    plt.show()
    return gini_scores


gini_scores = plot_lorenz_curves(df)

# Best individual model by Gini — used as the primary model for within-bin Lorenz
best_individual = max(gini_scores, key=gini_scores.get)
print(f"Individual Gini rankings: { {m: f'{g:.3f}' for m, g in sorted(gini_scores.items(), key=lambda x: -x[1])} }")
print(f"★  Best individual model (highest Gini): {best_individual}  ({gini_scores[best_individual]:.3f})")

## 3 · Pair Selection — CMI & JMI (5-fold CV)

Both metrics are computed in a single CV loop. The table shows **both a JMI rank and a CMI rank** for every pair. **Pairs are sorted and selected by highest JMI.**

| Metric | Formula | What it rewards | Role |
|---|---|---|---|
| **JMI** | `H(Y) − H(Y\|A,B)` | Total combined information the pair carries | **Selection & sort** |
| **CMI** | `[I(Y;B\|A) + I(Y;A\|B)] / 2` | Complementarity — new info each model adds beyond the other | Diagnostic reference |

**Why select by JMI?**  
JMI measures the pair's absolute discrimination ceiling — how much of the charge-off signal the two models can explain together. Selecting the highest-JMI pair maximises total predictive power.

**Why show CMI rank alongside?**  
CMI reveals whether the pair's power comes from two *complementary* models or from one dominant model carrying the other. A pair with JMI rank #1 but CMI rank #10 signals that both models are strong but largely redundant — useful context for model governance and vendor diversification decisions.

**Cross-validation:** bin boundaries are fit on the **training fold only** and applied to the held-out test fold to prevent in-sample inflation.

### Entropy — What It Is and How We Calculate It

**Entropy** measures *uncertainty*. In this pipeline we use **binary entropy** because our outcome `Y` (charge-off) is a 0/1 variable.

---

#### Binary entropy `H(p)`

For a population where a fraction `p` of accounts charge off:

$$H(p) = -p \log_2 p - (1-p) \log_2 (1-p) \quad \text{(bits)}$$

| `p` (charge-off rate) | `H(p)` | Interpretation |
|---|---|---|
| 0% or 100% | 0.0 bits | No uncertainty — outcome is certain |
| 50% | 1.0 bit | Maximum uncertainty — perfectly unpredictable |
| 25% (our portfolio) | ~0.81 bits | Moderate uncertainty |

The function `_h(p)` computes this directly. Edge cases `p = 0` and `p = 1` return `0.0` to avoid `log(0)`.

---

#### Conditional entropy `H(Y | A)` — uncertainty *after* knowing model A's bin

We discretise each model score into `N_BINS = 5` quantile bins. Given we know which bin an account falls in, the residual uncertainty is:

$$H(Y \mid A) = \sum_{a} \frac{n_a}{n} \cdot H\!\left(\bar{y}_a\right)$$

where:
- the sum runs over each bin `a` that model A produces
- $n_a$ = number of accounts in bin `a`
- $\bar{y}_a$ = charge-off rate inside bin `a` (the local `p`)
- $\frac{n_a}{n}$ = weight for that bin (accounts in bin / total accounts)

**Intuition:** if a model is predictive, accounts in its high-score bins will have very low charge-off rates (H ≈ 0) and accounts in its low-score bins will have high rates (H still approaches 0 in the extreme). A model with no predictive power leaves the charge-off rate the same in every bin, so H(Y|A) ≈ H(Y).

`_h_y_given_a(y, a_bins)` implements this for a single model.

---

#### Joint conditional entropy `H(Y | A, B)` — uncertainty after knowing *both* models' bins

$$H(Y \mid A, B) = \sum_{a,b} \frac{n_{ab}}{n} \cdot H\!\left(\bar{y}_{ab}\right)$$

Same idea, but now we condition on the cross-product of both models' bins (up to 25 cells). Accounts in the same (a, b) cell share a local charge-off rate $\bar{y}_{ab}$.

`_h_y_given_ab(y, a_bins, b_bins)` implements this via a groupby over both bin columns.

---

#### From entropy to JMI and CMI

| Quantity | Formula | Meaning |
|---|---|---|
| `H(Y)` | `_h(y.mean())` | Baseline uncertainty in the full portfolio |
| `H(Y\|A)` | `_h_y_given_a(y, a_bins)` | Remaining uncertainty after using model A alone |
| `H(Y\|B)` | `_h_y_given_a(y, b_bins)` | Remaining uncertainty after using model B alone |
| `H(Y\|A,B)` | `_h_y_given_ab(y, a_bins, b_bins)` | Remaining uncertainty after using *both* models |
| **JMI** | `H(Y) − H(Y\|A,B)` | Total uncertainty *removed* by the pair — the pair's discrimination ceiling |
| **CMI** | `½ · [(H(Y\|A) − H(Y\|A,B)) + (H(Y\|B) − H(Y\|A,B))]` | Average *new* information each model adds beyond the other |

All quantities are in **bits** (log base 2). A larger JMI = more charge-off signal captured. A larger CMI = the two models are more complementary (less redundant).

In [ ]:
# ── Entropy helpers ────────────────────────────────────────────────────────────
def _h(p: float) -> float:
    """Binary entropy in bits."""
    if p <= 0.0 or p >= 1.0:
        return 0.0
    return -p * np.log2(p) - (1.0 - p) * np.log2(1.0 - p)


def _make_cuts(scores, n_bins):
    """Quantile cut points with ±inf extension so no test value falls outside."""
    _, cuts = pd.qcut(scores, q=n_bins, retbins=True, duplicates="drop")
    cuts[0], cuts[-1] = -np.inf, np.inf
    return cuts


def _apply_cuts(scores, cuts):
    return np.array(pd.cut(scores, bins=cuts, labels=False), dtype=float)


def _h_y_given_a(y, a_bins):
    """H(Y | A) — conditional entropy given one model's bins."""
    n, h = len(y), 0.0
    for av in np.unique(a_bins[~np.isnan(a_bins)]):
        m = a_bins == av
        h += m.sum() / n * _h(y[m].mean())
    return h


def _h_y_given_ab(y, a_bins, b_bins):
    """H(Y | A, B) — residual entropy given both model bins."""
    n  = len(y)
    dt = pd.DataFrame({"y": y, "a": a_bins, "b": b_bins}).dropna()
    h  = 0.0
    for _, grp in dt.groupby(["a", "b"]):
        h += len(grp) / n * _h(grp["y"].mean())
    return h


def compute_pair_metrics_cv(df: pd.DataFrame, m1: str, m2: str) -> dict:
    """Compute symmetric CMI and JMI for a model pair via k-fold CV."""
    sub = df[[m1, m2, TARGET]].dropna().reset_index(drop=True)
    y, sa, sb = sub[TARGET].values, sub[m1].values, sub[m2].values
    kf = KFold(n_splits=N_FOLDS, shuffle=True, random_state=42)
    cmi_vals, jmi_vals = [], []
    for tr, te in kf.split(y):
        ca, cb  = _make_cuts(sa[tr], N_BINS), _make_cuts(sb[tr], N_BINS)
        ab, bb  = _apply_cuts(sa[te], ca),    _apply_cuts(sb[te], cb)
        h_y     = _h(y[te].mean())
        h_y_a   = _h_y_given_a(y[te], ab)
        h_y_b   = _h_y_given_a(y[te], bb)
        h_y_ab  = _h_y_given_ab(y[te], ab, bb)
        cmi_vals.append((max(0.0, h_y_a - h_y_ab) + max(0.0, h_y_b - h_y_ab)) / 2)
        jmi_vals.append(max(0.0, h_y - h_y_ab))
    return {
        "Pair":     f"{m1} + {m2}",
        "N_valid":  len(sub),
        "CMI_mean": round(np.mean(cmi_vals), 6),
        "CMI_std":  round(np.std(cmi_vals),  6),
        "JMI_mean": round(np.mean(jmi_vals), 6),
        "JMI_std":  round(np.std(jmi_vals),  6),
    }


print("Pair metric functions defined.")

In [ ]:
print(f"Computing CMI & JMI for all {len(list(combinations(MODELS, 2)))} pairs ({N_FOLDS}-fold CV) …")
pair_rows = [compute_pair_metrics_cv(df, m1, m2) for m1, m2 in combinations(MODELS, 2)]
pair_df   = (pd.DataFrame(pair_rows)
               .sort_values("JMI_mean", ascending=False)
               .reset_index(drop=True))
pair_df["JMI_Rank"] = range(1, len(pair_df) + 1)
pair_df["CMI_Rank"] = pair_df["CMI_mean"].rank(ascending=False, method="first").astype(int)

pair_df.to_csv("pair_rankings.csv", index=False)
print("Saved: pair_rankings.csv")
pair_df

In [ ]:
def plot_pair_table(pair_df: pd.DataFrame) -> None:
    display = pair_df[["JMI_Rank", "CMI_Rank", "Pair", "N_valid",
                        "JMI_mean", "JMI_std",
                        "CMI_mean", "CMI_std"]].copy()
    display.columns = ["JMI Rank", "CMI Rank", "Model Pair", "N Valid",
                       "JMI Mean (bits)", "JMI Std",
                       "CMI Mean (bits)", "CMI Std"]

    fig, ax = plt.subplots(figsize=(14, 4))
    ax.axis("off")
    tbl = ax.table(cellText=display.values, colLabels=display.columns,
                   cellLoc="center", loc="center")
    tbl.auto_set_font_size(False)
    tbl.set_fontsize(8.5)
    tbl.scale(1, 1.6)

    n_cols = len(display.columns)
    for j in range(n_cols):
        tbl[0, j].set_facecolor("#2c3e50")
        tbl[0, j].set_text_props(color="white", fontweight="bold")
    # JMI columns (0, 4, 5) → dark green; CMI columns (1, 6, 7) → dark blue
    for j in (0, 4, 5):
        tbl[0, j].set_facecolor("#145a32")
    for j in (1, 6, 7):
        tbl[0, j].set_facecolor("#1a5276")
    for j in range(n_cols):
        tbl[1, j].set_facecolor("#d5f5e3")
        tbl[1, j].set_text_props(fontweight="bold")
    for i in range(2, len(display) + 1):
        for j in range(n_cols):
            tbl[i, j].set_facecolor("#f8f9fa" if i % 2 == 0 else "white")

    ax.set_title(
        "Pair Ranking — sorted by JMI  (CMI rank shown for comparison)",
        fontsize=11, fontweight="bold", pad=12)
    plt.tight_layout()
    plt.savefig("pair_table.png", bbox_inches="tight")
    plt.show()


plot_pair_table(pair_df)

In [ ]:
best    = pair_df.iloc[0]   # highest JMI — selection criterion
best_m1, best_m2 = [m.strip() for m in best["Pair"].split("+")]

print(f"★  Best pair : {best['Pair']}")
print(f"   JMI mean  = {best['JMI_mean']:.6f} bits  ← selection criterion (total combined information)")
print(f"   CMI mean  = {best['CMI_mean']:.6f} bits  ← complementarity diagnostic (CMI rank #{best['CMI_Rank']})")
print()
print("Reading the table:")
print("  JMI ranks pairs by total combined discrimination power: how much of the")
print("  charge-off signal the two models can explain together.")
print("  The highest-JMI pair maximises the pair's absolute predictive ceiling.")
print()
print("  CMI rank (diagnostic): reveals whether the pair's power comes from two")
print("  complementary models or one dominant model carrying the other.")
print(f"  {best['Pair']} has JMI rank #1 but CMI rank #{best['CMI_Rank']} —")
print("  both models are individually strong but largely redundant (overlapping signals).")
print("  This is useful context for governance: consider whether vendor diversification")
print("  outweighs the marginal loss in CMI complementarity.")

### 3b · Within-Bin Lorenz Curves — Partner Models Inside Each Main-Model Quintile

The **best individual model by Gini** (`best_individual`) — not the JMI pair winner — splits the population into **5 quintile bins**. Within each bin, we plot a separate Lorenz curve for every candidate partner model.

This answers: **"Given that the best single model already puts an account in bin X, which partner model best separates the remaining charge-off risk inside that segment?"**

- **Primary model selection:** highest individual Gini from Section 2 (pure discrimination power, independent of any pair)
- Each of the 5 subplots = one quintile bin of the primary model (Bin 1 = safest, Bin 5 = riskiest)
- Bin labels reflect score direction: higher score = safer for FICO-style (+1); higher prob = riskier for PD models (−1, displayed on negated scale)
- Each coloured line = one partner model's Lorenz curve inside that bin
- Gini shown per curve; a flat diagonal = no further discrimination within the bin

In [ ]:
def plot_within_bin_lorenz(df: pd.DataFrame, main: str) -> None:
    """
    Split population into N_BINS quintile bins by `main` (best individual Gini model).
    For each bin, plot a Lorenz curve for every partner model.

    Bin labelling is direction-aware (after normalisation, higher score = safer):
      pd.qcut lowest quantile  = lowest normalised score = riskiest  → Bin N_BINS
      pd.qcut highest quantile = highest normalised score = safest   → Bin 1
    For PD models (SCORE_DIRECTION = -1) the column was negated in Section 1, so the
    same convention holds; the suptitle notes the original score direction.
    """
    partners    = [m for m in MODELS if m != main]
    colors      = ["#2c7bb6", "#d7191c", "#1a9641", "#fdae61", "#9e4fb5"]
    line_styles = ["-", "--", "-.", ":", (0, (3, 1, 1, 1))]

    direction = SCORE_DIRECTION.get(main, +1)
    score_note = (
        "higher score = safer (FICO / scorecard)"
        if direction == +1
        else "higher prob = riskier (PD model, displayed on negated scale)"
    )

    # Bin labels: pd.qcut assigns labels[0] to lowest quantile.
    # Since higher normalised score = safer, lowest quantile = riskiest = Bin N_BINS.
    bin_display = (
        [f"Bin {N_BINS}\n(Riskiest)"]
        + [f"Bin {i}" for i in range(N_BINS - 1, 1, -1)]
        + ["Bin 1\n(Safest)"]
    )

    sub_all = df[[TARGET, main] + partners].dropna(subset=[main]).copy()
    sub_all["_main_bin"] = pd.qcut(
        sub_all[main], q=N_BINS, labels=bin_display, duplicates="drop"
    )

    bin_labels = sub_all["_main_bin"].cat.categories.tolist()
    fig, axes  = plt.subplots(1, N_BINS, figsize=(18, 5), sharey=False)
    fig.suptitle(
        f"Within-Bin Lorenz Curves  —  primary model: {main}  (selected by highest Gini)\n"
        f"Score type: {score_note}  |  Bin 1 = safest, Bin {N_BINS} = riskiest",
        fontsize=10, fontweight="bold", y=1.03
    )

    for ax, bin_label in zip(axes, bin_labels):
        grp   = sub_all[sub_all["_main_bin"] == bin_label].copy()
        n_bin = len(grp)
        co_bin = grp[TARGET].mean()
        ax.set_title(f"{bin_label}\nn={n_bin:,}  CO={co_bin:.1%}", fontsize=8.5)

        for j, partner in enumerate(partners):
            sub = grp[[TARGET, partner]].dropna()
            if sub[TARGET].nunique() < 2 or len(sub) < 10:
                continue

            sub_s = sub.sort_values(partner, ascending=True).reset_index(drop=True)
            cum_accts = np.arange(1, len(sub_s) + 1) / len(sub_s)
            cum_co    = sub_s[TARGET].cumsum() / sub_s[TARGET].sum()
            # Negate partner score: higher normalised score = safer → negate for roc_auc_score
            gini = round(2 * roc_auc_score(sub_s[TARGET], -sub_s[partner]) - 1, 3)

            ax.plot(cum_accts, cum_co,
                    color=colors[j], linestyle=line_styles[j], linewidth=1.6,
                    label=f"{partner}  ({gini:.3f})")

        ax.plot([0, 1], [0, 1], "k--", linewidth=0.8, alpha=0.5)
        ax.set_xlim(0, 1); ax.set_ylim(0, 1)
        ax.xaxis.set_major_formatter(mticker.PercentFormatter(xmax=1))
        ax.yaxis.set_major_formatter(mticker.PercentFormatter(xmax=1))
        ax.tick_params(labelsize=7)
        ax.grid(True, alpha=0.25, linestyle="--")
        ax.legend(loc="lower right", fontsize=7, title="Partner (Gini)", title_fontsize=7)

    axes[0].set_ylabel(f"Cumulative % of '{TARGET}' Events", fontsize=8)
    for ax in axes:
        ax.set_xlabel("Cumulative % of Accounts", fontsize=8)

    plt.tight_layout()
    plt.savefig("lorenz_within_bins.png", bbox_inches="tight")
    plt.show()


plot_within_bin_lorenz(df, best_individual)

## 4 · Build 5×5 Score Grid

Each model score is independently split into **5 equal-population quintile tiers**:
- **Tier 1** = top quintile (highest scores) = safest accounts
- **Tier 5** = bottom quintile (lowest scores) = riskiest accounts

This produces **25 cells**, each reporting account volume, charge-off count, and observed CO rate.

In [ ]:
def build_score_grid(df: pd.DataFrame, m1: str, m2: str, n_buckets: int = 5):
    sub    = df[[m1, m2, TARGET]].dropna().copy()
    labels = list(range(n_buckets, 0, -1))   # label 1=safest (highest score), label 5=riskiest (lowest score)

    sub[f"{m1}_tier"], cuts_m1 = pd.qcut(sub[m1], q=n_buckets, labels=labels,
                                          retbins=True, duplicates="drop")
    sub[f"{m2}_tier"], cuts_m2 = pd.qcut(sub[m2], q=n_buckets, labels=labels,
                                          retbins=True, duplicates="drop")

    grid = (
        sub.groupby([f"{m1}_tier", f"{m2}_tier"], observed=True)
        .agg(N=(TARGET, "count"), ChargeOffs=(TARGET, "sum"))
        .reset_index()
    )
    grid["CO_Rate"] = (grid["ChargeOffs"] / grid["N"]).round(4)
    return grid, cuts_m1, cuts_m2


grid, cuts_m1, cuts_m2 = build_score_grid(df, best_m1, best_m2)
grid.to_csv("score_grid_raw.csv", index=False)
print("Saved: score_grid_raw.csv")

m1c, m2c = f"{best_m1}_tier", f"{best_m2}_tier"
pivot_co  = grid.pivot_table(index=m1c, columns=m2c, values="CO_Rate", aggfunc="mean")
pivot_co.index.name = f"{best_m1} ↓ / {best_m2} →"
print(f"\n'{TARGET}' Rate per cell  (Tier 1=Safest, Tier 5=Riskiest):")
pivot_co.round(3)

## 5 · Risk Tier Assignment — 1.5× Rule

The 25 cells are sorted by CO rate and partitioned into **5 named risk tiers** via exhaustive search.

**Constraint:** Each consecutive tier pair must satisfy:
$$\text{Avg CO Rate}(\text{Tier}_{k+1}) \geq 1.5 \times \text{Avg CO Rate}(\text{Tier}_k)$$

Rates are **accounts-weighted averages** across all cells in a tier.  
The winning partition **maximises the minimum achieved multiplier** across all boundaries.

In [ ]:
def _weighted_co(cells: pd.DataFrame) -> float:
    total_n = cells["N"].sum()
    return cells["ChargeOffs"].sum() / total_n if total_n > 0 else 0.0


def _find_best_split(cells, n_tiers, mult):
    nc = len(cells)
    best_bounds, best_min_mult = None, -1.0
    for splits in combinations(range(1, nc), n_tiers - 1):
        bounds = (0,) + splits + (nc,)
        rates  = [_weighted_co(cells.iloc[bounds[i]:bounds[i+1]]) for i in range(n_tiers)]
        if any(r == 0.0 for r in rates[:-1]):
            continue
        mults = [rates[i+1] / rates[i] for i in range(n_tiers - 1)]
        if all(m >= mult for m in mults):
            min_m = min(mults)
            if min_m > best_min_mult:
                best_min_mult, best_bounds = min_m, bounds
    return best_bounds


def assign_tiers(grid: pd.DataFrame, n_tiers: int = N_TIERS,
                 mult: float = MULT) -> tuple:
    cells = grid.dropna(subset=["CO_Rate"]).copy()
    cells["_orig_idx"] = cells.index
    cells = cells.sort_values("CO_Rate", ascending=True).reset_index(drop=True)

    bounds = _find_best_split(cells, n_tiers, mult)
    if bounds is None:
        print(f"  WARNING: No {mult}x partition found — falling back to equal-count split.")
        nc     = len(cells)
        chunk  = nc // n_tiers
        bounds = tuple([i * chunk for i in range(n_tiers)] + [nc])

    tier_col, summary_rows, co_cuts = np.empty(len(cells), dtype=object), [], []
    for i in range(n_tiers):
        grp  = cells.iloc[bounds[i]:bounds[i+1]]
        name = TIER_ORDER[i]
        tier_col[bounds[i]:bounds[i+1]] = name
        rate = _weighted_co(grp)
        summary_rows.append({
            "Risk_Tier":   name,
            "N_Cells":     len(grp),
            "Accounts":    int(grp["N"].sum()),
            "ChargeOffs":  int(grp["ChargeOffs"].sum()),
            "Avg_CO_Rate": round(rate, 4),
            "CO_Rate_Min": round(grp["CO_Rate"].min(), 4),
            "CO_Rate_Max": round(grp["CO_Rate"].max(), 4),
        })
        co_cuts.append(round(grp["CO_Rate"].max(), 4))

    cells["Risk_Tier"] = tier_col
    tier_map            = cells.set_index("_orig_idx")["Risk_Tier"]
    grid_tiered         = grid.copy()
    grid_tiered["Risk_Tier"] = tier_map.reindex(grid_tiered.index).values

    summary = pd.DataFrame(summary_rows)
    rates   = summary["Avg_CO_Rate"].values
    summary["Multiplier_vs_Prev"] = ["-"] + [
        f"{rates[i] / rates[i-1]:.2f}x" for i in range(1, n_tiers)
    ]
    summary["CO_Rate_Pct"]    = (summary["Avg_CO_Rate"] * 100).round(1).astype(str) + "%"
    total_co                  = summary["ChargeOffs"].sum()
    summary["Pct_of_All_COs"] = (
        (summary["ChargeOffs"] / total_co * 100).round(1).astype(str) + "%"
    )

    return grid_tiered, summary, co_cuts


print("Tier assignment functions defined.")

In [ ]:
grid_tiered, summary, co_cuts = assign_tiers(grid)
grid_tiered.to_csv("score_grid_tiered.csv", index=False)
summary.to_csv("risk_tier_summary.csv", index=False)
print("Saved: score_grid_tiered.csv, risk_tier_summary.csv")

display_cols = ["Risk_Tier", "N_Cells", "Accounts", "CO_Rate_Pct",
                "CO_Rate_Min", "CO_Rate_Max", "Multiplier_vs_Prev", "Pct_of_All_COs"]
summary[display_cols]

In [ ]:
print(f"Multiplier validation (must be ≥ {MULT}x):")
rates = summary["Avg_CO_Rate"].values
all_pass = True
for i in range(1, N_TIERS):
    ratio  = rates[i] / rates[i - 1]
    status = "✓" if ratio >= MULT else "✗  FAILS"
    if ratio < MULT:
        all_pass = False
    print(f"  {summary['Risk_Tier'].iloc[i-1]:12s} → "
          f"{summary['Risk_Tier'].iloc[i]:12s} : "
          f"{rates[i-1]:.1%} → {rates[i]:.1%}  ({ratio:.2f}x)  {status}")

print()
print("All tiers pass" if all_pass else "WARNING: some tiers fail the constraint")

## 6 · Score Grid Chart

Coloured 5×5 heatmap showing:
- **Cell annotation:** account count (N), charge-off count (CO), CO rate (%)
- **Cell colour:** risk tier (green → red)
- **Axis labels:** actual score cut points from the quintile bins
- **Right panel:** tier legend with CO rate ranges and tier-over-tier multipliers

In [ ]:
def _score_range_label(tier_label: int, cuts: np.ndarray, model: str = "") -> str:
    """
    Return a human-readable score range for a tier label (1–N_BINS, 1=safest).
    cuts from pd.qcut are in ascending order with ±inf sentinels at the edges.

    For SCORE_DIRECTION = -1 (PD) models the scores were negated before binning,
    so cuts are on the negated scale.  This function un-negates them for display
    so the label always shows the original score range.
    """
    direction = SCORE_DIRECTION.get(model, +1)
    idx = N_BINS - int(tier_label)   # N_BINS (not hardcoded 5)

    if direction == -1:
        lo = -cuts[idx + 1]
        hi = -cuts[idx]
        fmt = lambda v: f"{v:.2f}"
        if not np.isfinite(hi): return f"≥ {fmt(lo)}"
        if not np.isfinite(lo): return f"≤ {fmt(hi)}"
        return f"{fmt(lo)}–{fmt(hi)}"
    else:
        lo   = cuts[idx]
        hi   = cuts[idx + 1]
        lo_s = f"{int(lo)}" if np.isfinite(lo) else f"{int(cuts[1] - 1)}"
        hi_s = f"{int(hi)}" if np.isfinite(hi) else f"{int(cuts[-2] + 1)}"
        return f"{lo_s}–{hi_s}"


def plot_score_grid(grid_tiered: pd.DataFrame, m1: str, m2: str,
                    cuts_m1: np.ndarray, cuts_m2: np.ndarray,
                    summary: pd.DataFrame, co_cuts: list) -> None:

    m1c, m2c = f"{m1}_tier", f"{m2}_tier"

    fig = plt.figure(figsize=(13, 7))
    gs  = fig.add_gridspec(1, 2, width_ratios=[3, 1.1], wspace=0.35)
    ax  = fig.add_subplot(gs[0])
    ax2 = fig.add_subplot(gs[1])
    ax2.axis("off")

    tier_labels_int = list(range(1, N_BINS + 1))

    for _, row in grid_tiered.iterrows():
        m1_t  = int(row[m1c])
        m2_t  = int(row[m2c])
        tier  = row.get("Risk_Tier")
        color = TIER_PALETTE.get(tier, "#cccccc")

        x, y = m2_t - 1, m1_t - 1
        rect = plt.Rectangle((x, y), 1, 1, facecolor=color, edgecolor="white",
                              linewidth=2, zorder=1)
        ax.add_patch(rect)

        n_acc = int(row["N"])
        n_co  = int(row["ChargeOffs"])
        co_r  = row["CO_Rate"]

        ax.text(x + 0.5, y + 0.72, f"N = {n_acc:,}",
                ha="center", va="center", fontsize=7.5, color="black", zorder=2)
        ax.text(x + 0.5, y + 0.48, f"CO = {n_co:,}",
                ha="center", va="center", fontsize=7.5, color="black", zorder=2)
        ax.text(x + 0.5, y + 0.24, f"{co_r:.1%}",
                ha="center", va="center", fontsize=9,
                fontweight="bold", color="black", zorder=2)

    ax.set_xlim(0, N_BINS); ax.set_ylim(0, N_BINS)

    x_labels = [_score_range_label(t, cuts_m2, m2) for t in tier_labels_int]
    y_labels = [_score_range_label(t, cuts_m1, m1) for t in tier_labels_int]

    ax.set_xticks([i + 0.5 for i in range(N_BINS)])
    ax.set_xticklabels(
        [f"Tier {t}\n({x_labels[i]})" for i, t in enumerate(tier_labels_int)], fontsize=8)
    ax.set_yticks([i + 0.5 for i in range(N_BINS)])
    ax.set_yticklabels(
        [f"Tier {t}\n({y_labels[i]})" for i, t in enumerate(tier_labels_int)], fontsize=8)

    ax.set_xlabel(f"{m2} Score  →  Tier 1 = Safest, Tier {N_BINS} = Riskiest", fontsize=10)
    ax.set_ylabel(f"{m1} Score  →  Tier 1 = Safest, Tier {N_BINS} = Riskiest", fontsize=10)
    ax.set_title(f"Score Grid: {m1} × {m2}\n(volume · charge-offs · CO%  |  coloured by risk tier)",
                 fontsize=11, fontweight="bold")

    ax2.set_title("Risk Tier Key\n& CO Rate Cut Points", fontsize=10, fontweight="bold", pad=8)

    # ── Legend: auto-scale spacing so 5–7 tiers all fit cleanly ──────────────
    n_legend = N_TIERS
    step  = min(0.22, 0.97 / n_legend)   # compress automatically for 6+ tiers
    scale = step / 0.22                   # proportionally shrink text offsets

    y_pos = 0.97
    for tier in TIER_ORDER:
        row_s = summary[summary["Risk_Tier"] == tier]
        if row_s.empty:
            continue
        row_s = row_s.iloc[0]

        patch = mpatches.FancyBboxPatch(
            (0.02, y_pos - 0.07 * scale), 0.96, 0.07 * scale,
            boxstyle="round,pad=0.01",
            facecolor=TIER_PALETTE[tier], edgecolor="white", linewidth=1.5,
            transform=ax2.transAxes, zorder=2
        )
        ax2.add_patch(patch)
        ax2.text(0.50, y_pos - 0.035 * scale, tier,
                 transform=ax2.transAxes, ha="center", va="center",
                 fontsize=max(6, 9 * scale), fontweight="bold", color="black", zorder=3)

        co_lo    = f"{row_s['CO_Rate_Min']:.1%}"
        co_hi    = f"{row_s['CO_Rate_Max']:.1%}"
        mult_str = row_s["Multiplier_vs_Prev"]
        fs_small = max(5.5, 7.8 * scale)
        ax2.text(0.50, y_pos - 0.115 * scale, f"CO range: {co_lo} – {co_hi}",
                 transform=ax2.transAxes, ha="center", va="center",
                 fontsize=fs_small, color="#333333")
        ax2.text(0.50, y_pos - 0.155 * scale, f"Avg CO: {row_s['CO_Rate_Pct']}  |  {mult_str}",
                 transform=ax2.transAxes, ha="center", va="center",
                 fontsize=max(5.5, 7.5 * scale), color="#555555")
        ax2.text(0.50, y_pos - 0.195 * scale, f"{row_s['Accounts']:,} accts  ·  {row_s['Pct_of_All_COs']} of COs",
                 transform=ax2.transAxes, ha="center", va="center",
                 fontsize=max(5.5, 7.5 * scale), color="#555555")
        y_pos -= step

    # CO cut-point dividers between tiers
    y_pos2 = 0.97
    for idx in range(N_TIERS - 1):
        cutoff = co_cuts[idx]
        ax2.text(0.50, y_pos2 - step * (idx + 1) + 0.01 * scale,
                 f"── CO cut: {cutoff:.1%} ──",
                 transform=ax2.transAxes, ha="center", va="center",
                 fontsize=max(5, 7 * scale), color="#888888", style="italic")

    plt.savefig("score_grid_chart.png", bbox_inches="tight")
    plt.show()


print("Chart function defined.")

In [ ]:
plot_score_grid(grid_tiered, best_m1, best_m2, cuts_m1, cuts_m2, summary, co_cuts)

## 7 · Risk Tier Definitions

Summary table of all active risk tiers with their CO rate thresholds, account volumes, tier-over-tier multipliers, and assigned colour.  
This table is the authoritative reference for downstream policy rules, pricing tables, and reporting.

In [ ]:
def plot_tier_definition_table(summary: pd.DataFrame) -> None:
    """
    Render a formatted Risk Tier Definition table as a matplotlib figure.
    Columns: Tier | Name | Colour swatch | CO Rate Range | Avg CO Rate | Multiplier vs Prev | Accounts | % of All COs
    Rows auto-scale to N_TIERS (5, 6, or 7).
    """
    cols = [
        "Tier",
        "Risk Tier Name",
        "Colour",
        "CO Rate Range",
        "Avg CO Rate",
        "Multiplier\nvs Prev Tier",
        "Accounts",
        "% of All COs",
    ]

    rows = []
    for i, tier in enumerate(TIER_ORDER, start=1):
        row_s = summary[summary["Risk_Tier"] == tier]
        if row_s.empty:
            continue
        r = row_s.iloc[0]
        rows.append([
            str(i),
            tier,
            "",   # placeholder — filled with a coloured patch below
            f"{r['CO_Rate_Min']:.1%} – {r['CO_Rate_Max']:.1%}",
            r["CO_Rate_Pct"],
            r["Multiplier_vs_Prev"],
            f"{r['Accounts']:,}",
            r["Pct_of_All_COs"],
        ])

    n_rows = len(rows)
    fig_h  = max(2.5, 0.55 * n_rows + 1.2)
    fig, ax = plt.subplots(figsize=(14, fig_h))
    ax.axis("off")

    tbl = ax.table(
        cellText=rows,
        colLabels=cols,
        cellLoc="center",
        loc="center",
    )
    tbl.auto_set_font_size(False)
    tbl.set_fontsize(9)
    tbl.scale(1, 1.9)

    n_cols = len(cols)

    # Header row styling
    header_colors = {
        0: "#2c3e50", 1: "#2c3e50", 2: "#2c3e50",
        3: "#145a32", 4: "#145a32",
        5: "#6e2f8a",
        6: "#1a5276", 7: "#1a5276",
    }
    for j in range(n_cols):
        cell = tbl[0, j]
        cell.set_facecolor(header_colors.get(j, "#2c3e50"))
        cell.set_text_props(color="white", fontweight="bold")

    # Data rows: alternating background + colour swatch in col 2
    for i, (tier, row_data) in enumerate(zip(TIER_ORDER, rows), start=1):
        tier_color = TIER_PALETTE.get(tier, "#cccccc")
        row_bg     = "#f8f9fa" if i % 2 == 0 else "white"

        for j in range(n_cols):
            cell = tbl[i, j]
            if j == 2:
                # Colour swatch cell — fill with the tier colour, blank text
                cell.set_facecolor(tier_color)
                cell.get_text().set_text("")
            else:
                cell.set_facecolor(row_bg)
                if j == 0:
                    cell.set_text_props(fontweight="bold")

    # Column widths
    col_widths = [0.04, 0.14, 0.04, 0.14, 0.09, 0.13, 0.10, 0.10]
    for j, w in enumerate(col_widths):
        tbl.auto_set_column_width([j])
        for i in range(n_rows + 1):
            tbl[i, j].set_width(w)

    ax.set_title(
        f"Risk Tier Definitions  ({N_TIERS} Active Tiers  |  Minimum Multiplier: {MULT}×)",
        fontsize=12, fontweight="bold", pad=14,
    )

    note = (
        "Colour swatch reflects the cell colour used in the Score Grid chart.  "
        "CO Rate Range = min/max cell-level rates within the tier.  "
        "Multiplier vs Prev = Avg CO Rate ÷ Avg CO Rate of the preceding (safer) tier."
    )
    fig.text(0.5, 0.01, note, ha="center", fontsize=7.5, color="#555555",
             style="italic", wrap=True)

    plt.tight_layout(rect=[0, 0.04, 1, 1])
    plt.savefig("risk_tier_definitions.png", bbox_inches="tight")
    plt.show()
    print("Saved: risk_tier_definitions.png")


plot_tier_definition_table(summary)

## 8 · Risk Tier Assignment Rules

One row per risk tier, showing the **Model A and Model B score ranges** that map to it.  
Saved to `score_rules.csv` for use in decision engines or policy documents.

In [ ]:
def _tier_score_range(bins: list, cuts: np.ndarray, model: str) -> str:
    """
    Given a list of bin labels (1=safest) belonging to one tier,
    return the overall score range spanning all those bins.
    For +1 models: higher score = safer; for -1 (PD) models scores are un-negated.
    """
    direction = SCORE_DIRECTION.get(model, +1)
    idxs = [N_BINS - t for t in bins]          # index into cuts array
    lo   = min(cuts[i]     for i in idxs)
    hi   = max(cuts[i + 1] for i in idxs)

    if direction == +1:
        lo_s = f"{int(lo)}" if np.isfinite(lo) else f"{int(cuts[1] - 1)}"
        hi_s = f"{int(hi)}" if np.isfinite(hi) else f"{int(cuts[-2] + 1)}"
        return f"{lo_s} – {hi_s}"
    else:
        lo_orig, hi_orig = -hi, -lo
        fmt = lambda v: f"{v:.2f}"
        lo_s = fmt(lo_orig) if np.isfinite(lo_orig) else "—"
        hi_s = fmt(hi_orig) if np.isfinite(hi_orig) else "—"
        return f"{lo_s} – {hi_s}"


def _bin_label(bins: list) -> str:
    bins = sorted(bins)
    return f"Bin {bins[0]}" if len(bins) == 1 else f"Bins {bins[0]}–{bins[-1]}"


def plot_tier_rules(grid_tiered: pd.DataFrame, m1: str, m2: str,
                    cuts_m1: np.ndarray, cuts_m2: np.ndarray) -> pd.DataFrame:
    m1c, m2c = f"{m1}_tier", f"{m2}_tier"

    rows = []
    for tier in TIER_ORDER:
        cells = grid_tiered[grid_tiered["Risk_Tier"] == tier]
        if cells.empty:
            continue

        m1_bins = sorted(cells[m1c].astype(int).unique())
        m2_bins = sorted(cells[m2c].astype(int).unique())

        rows.append({
            "Risk Tier":            tier,
            f"{m1} Bins":           _bin_label(m1_bins),
            f"{m1} Score Range":    _tier_score_range(m1_bins, cuts_m1, m1),
            f"{m2} Bins":           _bin_label(m2_bins),
            f"{m2} Score Range":    _tier_score_range(m2_bins, cuts_m2, m2),
        })

    rules_df = pd.DataFrame(rows)
    rules_df.to_csv("score_rules.csv", index=False)

    # ── Render as table figure ────────────────────────────────────────────────
    n_rows  = len(rules_df)
    fig_h   = max(2.0, 0.6 * n_rows + 1.0)
    fig, ax = plt.subplots(figsize=(13, fig_h))
    ax.axis("off")

    tbl = ax.table(
        cellText=rules_df.values,
        colLabels=rules_df.columns,
        cellLoc="center", loc="center",
    )
    tbl.auto_set_font_size(False)
    tbl.set_fontsize(9.5)
    tbl.scale(1, 2.2)

    n_cols = len(rules_df.columns)
    header_colors = ["#2c3e50", "#1a5276", "#1a5276", "#145a32", "#145a32"]
    for j in range(n_cols):
        tbl[0, j].set_facecolor(header_colors[j])
        tbl[0, j].set_text_props(color="white", fontweight="bold")

    for i, tier in enumerate(rules_df["Risk Tier"], start=1):
        tier_color = TIER_PALETTE.get(tier, "#cccccc")
        for j in range(n_cols):
            cell = tbl[i, j]
            cell.set_facecolor(tier_color if j == 0 else "white")
            if j == 0:
                cell.set_text_props(fontweight="bold")

    ax.set_title(
        f"Risk Tier Assignment Rules  —  {m1} × {m2}",
        fontsize=12, fontweight="bold", pad=12,
    )

    plt.tight_layout()
    plt.savefig("score_rules.png", bbox_inches="tight")
    plt.show()
    print("Saved: score_rules.png | score_rules.csv")
    return rules_df


rules_df = plot_tier_rules(grid_tiered, best_m1, best_m2, cuts_m1, cuts_m2)
rules_df

## 9 · Output Files

All files are saved in the current working directory.

In [ ]:
import os

output_files = [
    ("lorenz_curves.png",          "Lorenz curves for all 5 models individually"),
    ("lorenz_within_bins.png",     "Within-bin Lorenz curves — partner models inside each main-model quintile"),
    ("pair_table.png",             "Pair ranking table — CMI & JMI side-by-side"),
    ("score_grid_chart.png",       "Coloured 5×5 score grid chart with volume, CO counts, and CO rates"),
    ("risk_tier_definitions.png",  "Risk tier definition table with colour swatches and CO rate thresholds"),
    ("score_rules.png",            "Risk tier assignment rules — one row per tier with score ranges"),
    ("pair_rankings.csv",          "Raw CMI & JMI scores for all pairs"),
    ("score_grid_raw.csv",         "25-cell grid with CO rates"),
    ("score_grid_tiered.csv",      "Grid with risk tier labels"),
    ("risk_tier_summary.csv",      "Tier-level summary with multipliers"),
    ("score_rules.csv",            "Risk tier assignment rules (bin numbers + score ranges per tier)"),
]

print(f"{'File':<45} {'Size':>10}  Description")
print("-" * 90)
for fname, desc in output_files:
    size = f"{os.path.getsize(fname):,} B" if os.path.exists(fname) else "NOT FOUND"
    print(f"{fname:<45} {size:>10}  {desc}")